Si vous générer du texte avec des RNN / FNN, vous pouvez utilisez un modèle word2vec pour générer les plongements des tokens à l‘entrée, ou alors réapprendre les plongements pendant l’entraînement avec une couche d’embedding. Vous pouvez aussi utiliser tf-idf même si cela est moins courant. Vous n’êtes pas obligé d’utiliser toutes les méthodes de plongements ici, choisissez-en une.

Pour classifier et générer du texte avec un transformer, je vous recommande la bibliothèque Transformers de HuggingFace (et notamment le Trainer pour l'entrainement des modèles : https://huggingface.co/docs/transformers/main/en/trainer). Vous pouvez utiliser un petit modèle disponible sur leur site. Ne prenez pas un modèle à plus d’un milliard de paramètres si vous n’avez pas de bons GPUs, ce qui m’intéresse est comment vous aborder un problème, pas d’avoir les meilleurs performances possibles.

## Introduction (RNN)

- Plongement des tokens : Word2vec
- Plogement des documents : TF-IDF

### Evaluation

- BLEU/ROUGE Score
- BERT Score

### Bechmark

| Model                               | BLEU   | ROUGE-L | BERT F1 | BERT P | BERT R |
|------------------------------------|--------|---------|---------|--------|--------|
| **GRU + Word2Vec**                 | 0.0229 | 0.1172  | 0.3737  | 0.3748 | 0.3718 |
| **LSTM + Word2Vec**                | 0.0247 | 0.1236  | 0.3764  | 0.3730 | 0.3790 |
| **Bi-GRU + Word2Vec**              | 0.0242 | 0.1271  | 0.3746  | 0.3709 | 0.3777 |
| **Bi-LSTM + Word2Vec**             | 0.0239 | 0.1180  | 0.3751  | 0.3702 | 0.3790 |


## Common code

In [3]:
import pandas as pd
from nltk.tokenize import word_tokenize

In [4]:
df_all = pd.read_csv("MovieDataThread.csv")
df = df_all.sample(n=1000, random_state=42) # Remove this line to use the entire dataset
display(df["Script"].head())


37439    Okay, go ahead.\n My name is Sidney Westerfeld...
11898    1\n A plague has descended upon Earth.\n In on...
24628    Hi, Leo.\n Say, kid, if these flowers ain't | ...
26980    1\n Under the light\n Of the full moon\n Shimm...
35203    Spend all day with us.\n There are two--\n par...
Name: Script, dtype: object

In [5]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [6]:
import re

def script_tokenize(text):
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\n", " NEWLINE_TOKEN ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip().split()

def custom_tokenize(text):
    first_tokenize = script_tokenize(text)
    entokenized = " ".join(first_tokenize)

    second_tokenize = word_tokenize(entokenized)
    return second_tokenize

In [7]:
from gensim.models import Word2Vec

# Tokeniser tous les scripts
tokenized_scripts = [custom_tokenize(script) for script in df_train["Script"]]

# Entraînement Word2Vec
w2v_model = Word2Vec(sentences=tokenized_scripts, vector_size=100, window=5, min_count=2, workers=4)

# Sauvegarder le modèle
w2v_model.save("word2vec_movie_scripts.model")

In [8]:
import torch

# Add padding (for too short sequences) and unknown tokens (for out-of-vocabulary words) by default
word_to_idx = {'<PAD>': 0, '<UNK>': 1}
for i, word in enumerate(w2v_model.wv.index_to_key):
    word_to_idx[word] = i + 2
idx_to_word = {i: w for w, i in word_to_idx.items()}

# Create the embedding matrix, where each row corresponds to a word in the vocabulary and each column corresponds to a dimension of the embedding
embedding_matrix = torch.zeros((len(word_to_idx), 100))
for word, i in word_to_idx.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = torch.tensor(w2v_model.wv[word])

In [9]:
def encode_sequence(tokens):
    return [word_to_idx.get(t, 1) for t in tokens]

def prepare_sequences(tokenized_texts, seq_len=20):
    X, y = [], []
    for tokens in tokenized_texts:
        encoded = encode_sequence(tokens)
        for i in range(len(encoded) - seq_len):
            X.append(encoded[i:i+seq_len])
            y.append(encoded[i+seq_len])
    return torch.tensor(X), torch.tensor(y)

# Limit the dataset to 50,000 sequences
# 1 sequence = 20 tokens
# X = 20 tokens, y = 21st token
X_train, y_train = prepare_sequences(tokenized_scripts, seq_len=20)
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
X_train = X_train[:50000]
y_train = y_train[:50000]

X_train shape: torch.Size([9174071, 20])
y_train shape: torch.Size([9174071])


In [10]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
import re
import string

def calculate_bleu(reference, candidate):
    """
    Calculate BLEU score (it measures the similarity between two sentences)
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The BLEU score
    """
    smoothing_function = SmoothingFunction().method4
    bleu_score = sentence_bleu([reference], candidate, smoothing_function=smoothing_function)
    return bleu_score

def calculate_rouge(reference, candidate):
    """
    Calculate ROUGE-L score (it calculates the longest common subsequence)
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The ROUGE score
    """
    rouge = Rouge()
    scores = rouge.get_scores(candidate, reference)
    return scores[0]['rouge-l']['f']

def preprocess_text(text):
    """
    Preprocess the text by removing punctuation and converting to lowercase
    :param text: The input text
    :return: The preprocessed text
    """
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = text.lower()
    return text

def calculate_scores(reference, candidate):
    """
    Calculate BLEU and ROUGE scores for the given reference and candidate sentences
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The BLEU and ROUGE scores
    """
    # Preprocess the text
    reference = preprocess_text(reference)
    candidate = preprocess_text(candidate)

    reference_tokens = word_tokenize(reference)
    candidate_tokens = word_tokenize(candidate)

    bleu_score = calculate_bleu(reference_tokens, candidate_tokens)
    rouge_score = calculate_rouge(reference, candidate)

    return bleu_score, rouge_score

def calculate_perplexity(model, inputs, targets, criterion):
    model.eval()
    with torch.no_grad():
        outputs = model(inputs)         # (1, seq_len, vocab_size)
        outputs = outputs.view(-1, outputs.size(-1))  # (seq_len, vocab_size)
        targets = targets.view(-1)   
        loss = criterion(outputs, targets)
    return torch.exp(loss).item()

## RNN (GRU + Word2vec)
### Class

In [11]:
import torch.nn as nn

# Define the RNN model
# The model is a simple GRU with an embedding layer (Word2Vec embeddings)
class ScriptRNN(nn.Module):
    def __init__(self, embedding_matrix, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False, padding_idx=word_to_idx['<PAD>'])
        self.rnn = nn.GRU(embedding_matrix.size(1), hidden_size, batch_first=True, num_layers=2, dropout=0.3)
        self.fc = nn.Linear(hidden_size, embedding_matrix.size(0))

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

### Train

In [12]:
import torch
from torch.utils.data import DataLoader, TensorDataset
# Check if MPS is available
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")

# Model, loss, optimizer
model = ScriptRNN(embedding_matrix, hidden_size=64).to(device)
criterion = torch.nn.CrossEntropyLoss(ignore_index=word_to_idx['<PAD>'])
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Dataloader
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

def train_model(model, train_loader, criterion, optimizer, num_epochs=10, patience=3, save_path="best_model.pth"):
    best_loss = float("inf")
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print("New best model saved.")
        else:
            patience_counter += 1
            print(f"No improvement. Patience {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

train_model(model, train_loader, criterion, optimizer, num_epochs=10, patience=3, save_path="best_script_rnn.pth")


/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch 1/10 | Loss: 5.3319
New best model saved.
Epoch 2/10 | Loss: 4.1974
New best model saved.
Epoch 3/10 | Loss: 3.9032
New best model saved.
Epoch 4/10 | Loss: 3.7171
New best model saved.
Epoch 5/10 | Loss: 3.5800
New best model saved.
Epoch 6/10 | Loss: 3.4606
New best model saved.
Epoch 7/10 | Loss: 3.3543
New best model saved.
Epoch 8/10 | Loss: 3.2570
New best model saved.
Epoch 9/10 | Loss: 3.1730
New best model saved.
Epoch 10/10 | Loss: 3.1017
New best model saved.


### Load Model

In [13]:
# Load model
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")


model = ScriptRNN(embedding_matrix, hidden_size=64).to(device)
model.load_state_dict(torch.load("best_script_rnn.pth", map_location=torch.device("cpu")))


<All keys matched successfully>

### Text generation

In [14]:
import torch.nn.functional as F

def generate_text(model, seed_text, word_to_idx, idx_to_word, max_len=50, seq_len=20, temperature=1.0, device="cpu"):
    model.eval()
    tokens = custom_tokenize(seed_text)
    generated = tokens.copy()
    for _ in range(max_len):
        # Context = last seq_len tokens of the generated text
        context = tokens[-seq_len:]
        input_indices = [word_to_idx.get(w, word_to_idx["<UNK>"]) for w in context]

        if len(input_indices) < seq_len:
            input_indices = [word_to_idx["<PAD>"]] * (seq_len - len(input_indices)) + input_indices
            
        input_tensor = torch.tensor([input_indices], dtype=torch.long).to(device)

        with torch.no_grad():
            logits = model(input_tensor)
            probs = F.softmax(logits[0] / temperature, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1).item()

        next_word = idx_to_word.get(next_idx, "<UNK>")

        if next_word == "<PAD>": break
        tokens.append(next_word)
        generated.append(next_word)
    return " ".join(generated)

### Evaluation

In [15]:
import time
from bert_score import BERTScorer

bleu_scores, rouge_scores = [], []
bert_f1, bert_precision, bert_recall = [], [], []

SCRIPT_LIMIT_TOKEN_SIZE = 50
GENERATE_LIMIT_TOKEN_SIZE = 50

scorer = BERTScorer(lang="en", model_type="roberta-base", rescale_with_baseline=True)

total = len(df_test)
i = 1

print("Generating text and calculating scores...")
for script in df_test["Script"]:
    print(f"Processing script {i}/{total}...")
    print("=" * 50)
    start_time = time.time()

    seed_tokens = custom_tokenize(script)[:SCRIPT_LIMIT_TOKEN_SIZE]
    seed_text = " ".join(seed_tokens)

    generated_text = generate_text(
        model, seed_text, word_to_idx, idx_to_word,
        max_len=GENERATE_LIMIT_TOKEN_SIZE,
        seq_len=SCRIPT_LIMIT_TOKEN_SIZE,
        device=device,
        temperature=0.8
    )

    generated_tokens = custom_tokenize(generated_text)
    new_tokens = generated_tokens[len(seed_tokens):]
    generated = " ".join(new_tokens)

    candidate = generated

    reference_tokens = custom_tokenize(script)
    reference = " ".join(reference_tokens)[len(seed_text):len(seed_text) + len(generated)]

    print(f"Seed:\n{seed_text}")
    print("-" * 50)
    print(f"Reference:\n{reference}")
    print("-" * 50)
    print(f"Generated:\n{generated}")
    print("-" * 50)
    print(f"Time taken: {time.time() - start_time:.2f} seconds")

    bleu_score, rouge_score = calculate_scores(reference, candidate)

    P, R, F1 = scorer.score([candidate], [reference])
    f1_score = F1.mean().item()
    precision_score = P.mean().item()
    recall_score = R.mean().item()

    bleu_scores.append(bleu_score)
    rouge_scores.append(rouge_score)

    bert_f1.append(f1_score)
    bert_precision.append(precision_score)
    bert_recall.append(recall_score)

    with open(f"rnn-GRU-{SCRIPT_LIMIT_TOKEN_SIZE}-{GENERATE_LIMIT_TOKEN_SIZE}.csv", "a") as f:
        f.write(f"{seed_text};{candidate};{reference};{bleu_score};{rouge_score};{f1_score};{precision_score};{recall_score}\n")

    i += 1


print("Average BLEU Score :", sum(bleu_scores) / len(bleu_scores))
print("Average ROUGE Score :", sum(rouge_scores) / len(rouge_scores))
print("Average BERT F1 Score :", sum(bert_f1) / len(bert_f1))
print("Average BERT Precision Score :", sum(bert_precision) / len(bert_precision))
print("Average BERT Recall Score :", sum(bert_recall) / len(bert_recall))

/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/transformers/utils/generic.py:260: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Generating text and calculating scores...
Processing script 1/200...
Seed:
1 NEWLINE_TOKEN Delhi flight ! NEWLINE_TOKEN Sorry , Ma'am . NEWLINE_TOKEN Boarding is closed . NEWLINE_TOKEN Listen , I know ! NEWLINE_TOKEN I am so sorry , I got delayed . NEWLINE_TOKEN Just do something . NEWLINE_TOKEN Please help me . NEWLINE_TOKEN Sorry , Ma'am . NEWLINE_TOKEN You should have
--------------------------------------------------
Reference:
 been on time . NEWLINE_TOKEN I know , I am so sorry ! NEWLINE_TOKEN I got stuck in traffic . NEWLINE_TOKEN You know how it is . NEWLINE_TOKEN Just , please ! Can you do this ? NEWLINE_TOKEN Okay , hold on . NEWLINE_TOKEN Thank you . NEWLINE_TOKEN Window seat 
--------------------------------------------------
Generated:
like the phone Not . NEWLINE_TOKEN I 'll feel you , just ... NEWLINE_TOKEN I 've never should known NEWLINE_TOKEN that I think then NEWLINE_TOKEN my may be you , okay ? NEWLINE_TOKEN The least you 're fucking since I love you . NEWLINE_TOKEN

## RNN (LSTM + Word2vec) with tensorflow

https://www.tensorflow.org/text/tutorials/text_generation

### Model

In [12]:
import torch

embedding_matrix = torch.zeros((len(word_to_idx), 100))

for word, i in word_to_idx.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = torch.tensor(w2v_model.wv[word])


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ScriptLSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()

        vocab_size, embedding_dim = embedding_matrix.size()

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix, freeze=False, padding_idx=word_to_idx['<PAD>']
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)          
        _, (hidden, _) = self.lstm(embedded)   
        last_hidden = hidden[-1]                 
        output = self.fc(last_hidden)           
        return output


### Train

In [25]:
from torch.utils.data import DataLoader, TensorDataset

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")

model = ScriptLSTM(embedding_matrix, hidden_dim=64).to(device)
criterion = torch.nn.CrossEntropyLoss(ignore_index=word_to_idx['<PAD>'])
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [26]:

def train_model(model, train_loader, criterion, optimizer, num_epochs=10, patience=3, save_path="best_script_lstm.pth"):
    best_loss = float("inf")
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print("New model saved.")
        else:
            patience_counter += 1
            print(f"No improvement. Patience {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

train_model(model, train_loader, criterion, optimizer, num_epochs=10, patience=3, save_path="best_script_lstm.pth")


Epoch 1/10 | Loss: 5.5152
New model saved.
Epoch 2/10 | Loss: 4.4836
New model saved.
Epoch 3/10 | Loss: 4.1672
New model saved.
Epoch 4/10 | Loss: 3.9653
New model saved.
Epoch 5/10 | Loss: 3.8142
New model saved.
Epoch 6/10 | Loss: 3.6931
New model saved.
Epoch 7/10 | Loss: 3.5892
New model saved.
Epoch 8/10 | Loss: 3.4984
New model saved.
Epoch 9/10 | Loss: 3.4194
New model saved.
Epoch 10/10 | Loss: 3.3459
New model saved.


### Load model

In [27]:
model.load_state_dict(torch.load("best_script_lstm.pth", map_location=torch.device("cpu")))

<All keys matched successfully>

### Text generation

In [28]:
import torch
import torch.nn.functional as F

def generate_text_lstm(model, seed_text, word_to_idx, idx_to_word,
                       max_len=50, seq_len=20, temperature=1.0, device="cpu"):
    model.eval()
    tokens = custom_tokenize(seed_text)
    generated = tokens.copy()

    for _ in range(max_len):
        context = tokens[-seq_len:]
        input_indices = [word_to_idx.get(w, word_to_idx["<UNK>"]) for w in context]

        if len(input_indices) < seq_len:
            input_indices = [word_to_idx["<PAD>"]] * (seq_len - len(input_indices)) + input_indices

        input_tensor = torch.tensor([input_indices], dtype=torch.long).to(device)

        with torch.no_grad():
            logits = model(input_tensor)
            probs = F.softmax(logits[0] / temperature, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1).item()

        next_word = idx_to_word.get(next_idx, "<UNK>")
        if next_word == "<PAD>":
            break

        tokens.append(next_word)
        generated.append(next_word)

    return " ".join(generated)


### Scoring

In [29]:
import time
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
from bert_score import BERTScorer
from nltk.tokenize import word_tokenize
import re
import string

def preprocess_text(text):
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = text.lower()
    return text

def calculate_bleu(reference, candidate):
    smoothing = SmoothingFunction().method4
    return sentence_bleu([reference], candidate, smoothing_function=smoothing)

def calculate_rouge(reference, candidate):
    rouge = Rouge()
    scores = rouge.get_scores(candidate, reference)
    return scores[0]['rouge-l']['f']

def calculate_scores(reference, candidate):
    reference = preprocess_text(reference)
    candidate = preprocess_text(candidate)

    reference_tokens = word_tokenize(reference)
    candidate_tokens = word_tokenize(candidate)

    bleu_score = calculate_bleu(reference_tokens, candidate_tokens)
    rouge_score = calculate_rouge(reference, candidate)

    return bleu_score, rouge_score


In [30]:
from bert_score import BERTScorer
import time

bleu_scores, rouge_scores = [], []
bert_f1, bert_precision, bert_recall = [], [], []

SCRIPT_LIMIT_TOKEN_SIZE = 50
GENERATE_LIMIT_SIZE = 50

scorer = BERTScorer(lang="en", model_type="roberta-base", rescale_with_baseline=True)

total = len(df_test)
i = 1

print("Generating text and calculating scores...")
for script in df_test["Script"]:
    print(f"Script {i}/{total}")

    seed_tokens = custom_tokenize(script)[:SCRIPT_LIMIT_TOKEN_SIZE]
    seed_text = " ".join(seed_tokens)

    generated_text = generate_text_lstm(
        model=model,
        seed_text=seed_text,
        word_to_idx=word_to_idx,
        idx_to_word=idx_to_word,
        max_len=GENERATE_LIMIT_SIZE,
        seq_len=SCRIPT_LIMIT_TOKEN_SIZE,
        device=device,
        temperature=0.8
    )

    generated_tokens = custom_tokenize(generated_text)
    new_tokens = generated_tokens[len(seed_tokens):]
    generated = " ".join(new_tokens)

    reference_tokens = custom_tokenize(script)
    reference = " ".join(reference_tokens)[len(seed_text):len(seed_text) + len(generated)]

    bleu, rouge = calculate_scores(reference, generated)
    bleu_scores.append(bleu)
    rouge_scores.append(rouge)

    P, R, F1 = scorer.score([generated], [reference])
    f1_score = F1.mean().item()
    precision_score = P.mean().item()
    recall_score = R.mean().item()

    bert_f1.append(f1_score)
    bert_precision.append(precision_score)
    bert_recall.append(recall_score)

    with open(f"rnn-LSTM-{SCRIPT_LIMIT_TOKEN_SIZE}-{GENERATE_LIMIT_SIZE}.csv", "a") as f:
        f.write(f"{seed_text};{generated};{reference};{bleu};{rouge};{f1_score};{precision_score};{recall_score}\n")

    i += 1

print("=" * 50)
print(f"Average BLEU Score : {sum(bleu_scores)/len(bleu_scores):.4f}")
print(f"Average ROUGE Score : {sum(rouge_scores)/len(rouge_scores):.4f}")
print(f"Average BERT F1 Score : {sum(bert_f1)/len(bert_f1):.4f}")
print(f"Average BERT Precision Score : {sum(bert_precision)/len(bert_precision):.4f}")
print(f"Average BERT Recall Score : {sum(bert_recall)/len(bert_recall):.4f}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Generating text and calculating scores...
Script 1/200
Script 2/200
Script 3/200
Script 4/200
Script 5/200
Script 6/200
Script 7/200
Script 8/200
Script 9/200
Script 10/200
Script 11/200
Script 12/200
Script 13/200
Script 14/200
Script 15/200
Script 16/200
Script 17/200
Script 18/200
Script 19/200
Script 20/200
Script 21/200
Script 22/200
Script 23/200
Script 24/200
Script 25/200
Script 26/200
Script 27/200
Script 28/200
Script 29/200
Script 30/200
Script 31/200
Script 32/200
Script 33/200
Script 34/200
Script 35/200
Script 36/200
Script 37/200
Script 38/200
Script 39/200
Script 40/200
Script 41/200
Script 42/200
Script 43/200
Script 44/200
Script 45/200
Script 46/200
Script 47/200
Script 48/200
Script 49/200
Script 50/200
Script 51/200
Script 52/200
Script 53/200
Script 54/200
Script 55/200
Script 56/200
Script 57/200
Script 58/200
Script 59/200
Script 60/200
Script 61/200
Script 62/200
Script 63/200
Script 64/200
Script 65/200
Script 66/200
Script 67/200
Script 68/200
Script 69/200
S

## RNN (GRU Bidirectional + Word2vec)
### Model

In [31]:
import torch.nn as nn

class ScriptRNNBI(nn.Module):
    def __init__(self, embedding_matrix, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)

        self.rnn = nn.GRU(
            input_size=embedding_matrix.size(1),
            hidden_size=hidden_size,
            batch_first=True,
            num_layers=2,
            dropout=0.2,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_size * 2, embedding_matrix.size(0))

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])


### Train

In [32]:
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")

model_bi = ScriptRNNBI(embedding_matrix, hidden_size=64).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=word_to_idx['<PAD>'])
optimizer = torch.optim.Adam(model_bi.parameters(), lr=0.001)

from torch.utils.data import DataLoader, TensorDataset
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

def train_model(model, train_loader, criterion, optimizer, num_epochs=10, patience=3, save_path="best_model.pth"):
    best_loss = float("inf")
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print("New best model saved.")
        else:
            patience_counter += 1
            print(f"No improvement. Patience {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

train_model(model_bi, train_loader, criterion, optimizer,
            num_epochs=10, patience=3, save_path="best_script_rnn_bi.pth")

Epoch 1/10 | Loss: 4.9257
New best model saved.
Epoch 2/10 | Loss: 3.9298
New best model saved.
Epoch 3/10 | Loss: 3.6176
New best model saved.
Epoch 4/10 | Loss: 3.3791
New best model saved.
Epoch 5/10 | Loss: 3.1807
New best model saved.
Epoch 6/10 | Loss: 3.0055
New best model saved.
Epoch 7/10 | Loss: 2.8530
New best model saved.
Epoch 8/10 | Loss: 2.7150
New best model saved.
Epoch 9/10 | Loss: 2.5887
New best model saved.
Epoch 10/10 | Loss: 2.4729
New best model saved.


### Load model

In [33]:
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")

model_bi = ScriptRNNBI(embedding_matrix, hidden_size=64).to(device)


model_bi.load_state_dict(torch.load("best_script_rnn_bi.pth"))
model_bi.eval()

import torch.nn.functional as F

def generate_text(model, seed_text, word_to_idx, idx_to_word, max_len=50, seq_len=20, temperature=1.0, device="cpu"):
    model.eval()
    tokens = custom_tokenize(seed_text)
    generated = tokens.copy()
    for _ in range(max_len):
        # Context = last seq_len tokens of the generated text
        context = tokens[-seq_len:]
        input_indices = [word_to_idx.get(w, word_to_idx["<UNK>"]) for w in context]

        if len(input_indices) < seq_len:
            input_indices = [word_to_idx["<PAD>"]] * (seq_len - len(input_indices)) + input_indices
            
        input_tensor = torch.tensor([input_indices], dtype=torch.long).to(device)

        with torch.no_grad():
            logits = model(input_tensor)
            probs = F.softmax(logits[0] / temperature, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1).item()

        next_word = idx_to_word.get(next_idx, "<UNK>")

        if next_word == "<PAD>": break
        tokens.append(next_word)
        generated.append(next_word)
    return " ".join(generated)

generated_text = generate_text(
    model=model_bi,
    seed_text="What's going on?",
    word_to_idx=word_to_idx,
    idx_to_word=idx_to_word,
    max_len=50,
    seq_len=20,
    device=device,
    temperature=0.8
)

print("Texte généré :\n", generated_text)


Texte généré :
 What 's going on ? NEWLINE_TOKEN You 're not ready . NEWLINE_TOKEN What 's up ? NEWLINE_TOKEN Are you okay ? NEWLINE_TOKEN No ! No , stop , you know ? NEWLINE_TOKEN I 'm sorry . Well , my God . NEWLINE_TOKEN Who 's your first name ? NEWLINE_TOKEN Oh , I do n't know


### Evaluation

In [34]:
from bert_score import BERTScorer

bleu_scores, rouge_scores = [], []
bert_f1, bert_precision, bert_recall = [], [], []


SCRIPT_LIMIT_TOKEN_SIZE = 50
GENERATE_LIMIT_SIZE = 50

scorer = BERTScorer(lang="en", model_type="roberta-base", rescale_with_baseline=True)

total = len(df_test)
i = 1

print("Generating text and calculating scores...")
for script in df_test["Script"]:
    print(f"Script {i}/{total}")
    seed_tokens = custom_tokenize(script)[:SCRIPT_LIMIT_TOKEN_SIZE]
    seed_text = " ".join(seed_tokens)

    generated_text = generate_text(
        model=model_bi,
        seed_text=seed_text,
        word_to_idx=word_to_idx,
        idx_to_word=idx_to_word,
        max_len=GENERATE_LIMIT_SIZE,
        seq_len=SCRIPT_LIMIT_TOKEN_SIZE,
        device=device,
        temperature=0.8
    )

    generated_tokens = custom_tokenize(generated_text)
    new_tokens = generated_tokens[len(seed_tokens):]
    generated = " ".join(new_tokens)

    reference_tokens = custom_tokenize(script)
    reference = " ".join(reference_tokens)[len(seed_text):len(seed_text) + len(generated)]

    # Scores
    bleu, rouge = calculate_scores(reference, generated)
    bleu_scores.append(bleu)
    rouge_scores.append(rouge)

    P, R, F1 = scorer.score([generated], [reference])
    f1_score = F1.mean().item()
    precision_score = P.mean().item()
    recall_score = R.mean().item()

    bert_f1.append(f1_score)
    bert_precision.append(precision_score)
    bert_recall.append(recall_score)
    
    with open(f"rnn-GRU-bi-{SCRIPT_LIMIT_TOKEN_SIZE}-{GENERATE_LIMIT_SIZE}.csv", "a") as f:
        f.write(f"{seed_text};{generated};{reference};{bleu};{rouge};{f1_score};{precision_score};{recall_score}\n")

    i += 1

# Moyennes
print("=" * 50)
print(f"Average BLEU Score : {sum(bleu_scores)/len(bleu_scores):.4f}")
print(f"Average ROUGE Score : {sum(rouge_scores)/len(rouge_scores):.4f}")
print(f"Average BERT F1 Score : {sum(bert_f1)/len(bert_f1):.4f}")
print(f"Average BERT Precision Score : {sum(bert_precision)/len(bert_precision):.4f}")
print(f"Average BERT Recall Score : {sum(bert_recall)/len(bert_recall):.4f}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Generating text and calculating scores...
Script 1/200
Script 2/200
Script 3/200
Script 4/200
Script 5/200
Script 6/200
Script 7/200
Script 8/200
Script 9/200
Script 10/200
Script 11/200
Script 12/200
Script 13/200
Script 14/200
Script 15/200
Script 16/200
Script 17/200
Script 18/200
Script 19/200
Script 20/200
Script 21/200
Script 22/200
Script 23/200
Script 24/200
Script 25/200
Script 26/200
Script 27/200
Script 28/200
Script 29/200
Script 30/200
Script 31/200
Script 32/200
Script 33/200
Script 34/200
Script 35/200
Script 36/200
Script 37/200
Script 38/200
Script 39/200
Script 40/200
Script 41/200
Script 42/200
Script 43/200
Script 44/200
Script 45/200
Script 46/200
Script 47/200
Script 48/200
Script 49/200
Script 50/200
Script 51/200
Script 52/200
Script 53/200
Script 54/200
Script 55/200
Script 56/200
Script 57/200
Script 58/200
Script 59/200
Script 60/200
Script 61/200
Script 62/200
Script 63/200
Script 64/200
Script 65/200
Script 66/200
Script 67/200
Script 68/200
Script 69/200
S

## RNN (LSTM Bidirectional + Word2vec)
### Model

In [35]:
import torch.nn as nn

class ScriptRNNBI_LSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)

        self.rnn = nn.LSTM(
            input_size=embedding_matrix.size(1),
            hidden_size=hidden_size,
            batch_first=True,
            num_layers=2,
            dropout=0.2,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_size * 2, embedding_matrix.size(0))

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])


### Train

In [36]:
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")

model_bi = ScriptRNNBI_LSTM(embedding_matrix, hidden_size=64).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=word_to_idx['<PAD>'])
optimizer = torch.optim.Adam(model_bi.parameters(), lr=0.001)

from torch.utils.data import DataLoader, TensorDataset
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

def train_model(model, train_loader, criterion, optimizer, num_epochs=10, patience=3, save_path="best_model.pth"):
    best_loss = float("inf")
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print("New best model saved.")
        else:
            patience_counter += 1
            print(f"No improvement. Patience {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

train_model(model_bi, train_loader, criterion, optimizer,
            num_epochs=10, patience=3, save_path="best_script_rnn_bi_lstm.pth")

Epoch 1/10 | Loss: 5.0621
New best model saved.
Epoch 2/10 | Loss: 4.0801
New best model saved.
Epoch 3/10 | Loss: 3.7890
New best model saved.
Epoch 4/10 | Loss: 3.5786
New best model saved.
Epoch 5/10 | Loss: 3.4006
New best model saved.
Epoch 6/10 | Loss: 3.2501
New best model saved.
Epoch 7/10 | Loss: 3.1083
New best model saved.
Epoch 8/10 | Loss: 2.9825
New best model saved.
Epoch 9/10 | Loss: 2.8619
New best model saved.
Epoch 10/10 | Loss: 2.7521
New best model saved.


### Load model

In [37]:
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")

model_bi_lstm = ScriptRNNBI_LSTM(embedding_matrix, hidden_size=64).to(device)


model_bi_lstm.load_state_dict(torch.load("best_script_rnn_bi_lstm.pth"))
model_bi_lstm.eval()

import torch.nn.functional as F

def generate_text(model, seed_text, word_to_idx, idx_to_word, max_len=50, seq_len=20, temperature=1.0, device="cpu"):
    model.eval()
    tokens = custom_tokenize(seed_text)
    generated = tokens.copy()
    for _ in range(max_len):
        # Context = last seq_len tokens of the generated text
        context = tokens[-seq_len:]
        input_indices = [word_to_idx.get(w, word_to_idx["<UNK>"]) for w in context]

        if len(input_indices) < seq_len:
            input_indices = [word_to_idx["<PAD>"]] * (seq_len - len(input_indices)) + input_indices
            
        input_tensor = torch.tensor([input_indices], dtype=torch.long).to(device)

        with torch.no_grad():
            logits = model(input_tensor)
            probs = F.softmax(logits[0] / temperature, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1).item()

        next_word = idx_to_word.get(next_idx, "<UNK>")

        if next_word == "<PAD>": break
        tokens.append(next_word)
        generated.append(next_word)
    return " ".join(generated)

generated_text = generate_text(
    model=model_bi_lstm,
    seed_text="What's going on?",
    word_to_idx=word_to_idx,
    idx_to_word=idx_to_word,
    max_len=50,
    seq_len=20,
    device=device,
    temperature=0.8
)

print("Texte généré :\n", generated_text)


Texte généré :
 What 's going on ? NEWLINE_TOKEN Do n't make it off , NEWLINE_TOKEN the fuck up , you 're calling me NEWLINE_TOKEN when you 're not tourists , NEWLINE_TOKEN but I 'm not racist . NEWLINE_TOKEN You 're doing this really ? NEWLINE_TOKEN - Okay , look , Grandma . NEWLINE_TOKEN I 'm going to


### Evaluation

In [38]:
from bert_score import BERTScorer

bleu_scores, rouge_scores = [], []
bert_f1, bert_precision, bert_recall = [], [], []

SCRIPT_LIMIT_TOKEN_SIZE = 50
GENERATE_LIMIT_SIZE = 50

scorer = BERTScorer(lang="en", model_type="roberta-base", rescale_with_baseline=True)

total = len(df_test)
i = 1

print("Evaluation of ScriptRNNBI (LSTM bidirectionnel)...")
for script in df_test["Script"]:
    print(f"Script {i}/{total}")
    seed_tokens = custom_tokenize(script)[:SCRIPT_LIMIT_TOKEN_SIZE]
    seed_text = " ".join(seed_tokens)

    generated_text = generate_text(
        model=model_bi_lstm,
        seed_text=seed_text,
        word_to_idx=word_to_idx,
        idx_to_word=idx_to_word,
        max_len=GENERATE_LIMIT_SIZE,
        seq_len=SCRIPT_LIMIT_TOKEN_SIZE,
        device=device,
        temperature=0.8
    )

    generated_tokens = custom_tokenize(generated_text)
    new_tokens = generated_tokens[len(seed_tokens):]
    generated = " ".join(new_tokens)

    reference_tokens = custom_tokenize(script)
    reference = " ".join(reference_tokens)[len(seed_text):len(seed_text) + len(generated)]

    # Scores
    bleu, rouge = calculate_scores(reference, generated)
    bleu_scores.append(bleu)
    rouge_scores.append(rouge)

    P, R, F1 = scorer.score([generated], [reference])
    f1_score = F1.mean().item()
    precision_score = P.mean().item()
    recall_score = R.mean().item()

    bert_f1.append(f1_score)
    bert_precision.append(precision_score)
    bert_recall.append(recall_score)
    
    with open(f"rnn-LSTM-bi-{SCRIPT_LIMIT_TOKEN_SIZE}-{GENERATE_LIMIT_SIZE}.csv", "a") as f:
        f.write(f"{seed_text};{generated};{reference};{bleu};{rouge};{f1_score};{precision_score};{recall_score}\n")

    i += 1

# Means
print("=" * 50)
print(f"Average BLEU Score : {sum(bleu_scores)/len(bleu_scores):.4f}")
print(f"Average ROUGE Score : {sum(rouge_scores)/len(rouge_scores):.4f}")
print(f"Average BERT F1 Score : {sum(bert_f1)/len(bert_f1):.4f}")
print(f"Average BERT Precision Score : {sum(bert_precision)/len(bert_precision):.4f}")
print(f"Average BERT Recall Score : {sum(bert_recall)/len(bert_recall):.4f}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluation of ScriptRNNBI (LSTM bidirectionnel)...
Script 1/200
Script 2/200
Script 3/200
Script 4/200
Script 5/200
Script 6/200
Script 7/200
Script 8/200
Script 9/200
Script 10/200
Script 11/200
Script 12/200
Script 13/200
Script 14/200
Script 15/200
Script 16/200
Script 17/200
Script 18/200
Script 19/200
Script 20/200
Script 21/200
Script 22/200
Script 23/200
Script 24/200
Script 25/200
Script 26/200
Script 27/200
Script 28/200
Script 29/200
Script 30/200
Script 31/200
Script 32/200
Script 33/200
Script 34/200
Script 35/200
Script 36/200
Script 37/200
Script 38/200
Script 39/200
Script 40/200
Script 41/200
Script 42/200
Script 43/200
Script 44/200
Script 45/200
Script 46/200
Script 47/200
Script 48/200
Script 49/200
Script 50/200
Script 51/200
Script 52/200
Script 53/200
Script 54/200
Script 55/200
Script 56/200
Script 57/200
Script 58/200
Script 59/200
Script 60/200
Script 61/200
Script 62/200
Script 63/200
Script 64/200
Script 65/200
Script 66/200
Script 67/200
Script 68/200
Script